# Analisis Klaim Pengeluaran

Notebook ini menunjukkan cara membuat agen yang menggunakan plugin untuk memproses pengeluaran perjalanan dari gambar struk lokal, mengirim email klaim pengeluaran, dan memvisualisasikan data pengeluaran menggunakan diagram lingkaran. Agen memilih fungsi secara dinamis berdasarkan konteks tugas.

Langkah-langkah:
1. Agen OCR memproses gambar struk lokal dan mengekstrak data pengeluaran perjalanan.
2. Agen Email membuat email klaim pengeluaran.

### Contoh skenario pengeluaran perjalanan:
Bayangkan Anda seorang karyawan yang melakukan perjalanan untuk rapat bisnis di kota lain. Perusahaan Anda memiliki kebijakan untuk mengganti semua pengeluaran terkait perjalanan yang wajar. Berikut adalah rincian potensi pengeluaran perjalanan:
- Transportasi:
Tiket pesawat pulang-pergi dari kota asal Anda ke kota tujuan.
Taksi atau layanan ride-hailing ke dan dari bandara.
Transportasi lokal di kota tujuan (seperti transit umum, mobil sewaan, atau taksi).

- Akomodasi:
Menginap di hotel selama tiga malam di hotel bisnis kelas menengah dekat tempat rapat.

- Makanan:
Tunjangan makanan harian untuk sarapan, makan siang, dan makan malam, berdasarkan kebijakan per diem perusahaan.

- Pengeluaran Lainnya:
Biaya parkir di bandara.
Biaya akses internet di hotel.
Tips atau biaya layanan kecil.

- Dokumentasi:
Anda menyerahkan semua struk (penerbangan, taksi, hotel, makanan, dll.) dan laporan pengeluaran lengkap untuk penggantian biaya.


## Impor pustaka yang diperlukan

Impor pustaka dan modul yang diperlukan untuk notebook.


In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import base64
import dotenv
from typing import Annotated, List

from pydantic import BaseModel, Field

from agent_framework import tool, AgentResponseUpdate, WorkflowBuilder
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

dotenv.load_dotenv()

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

In [ ]:
# Create the Azure AI Foundry client
client = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

 ## Definisikan Model Pengeluaran

 Buat model Pydantic untuk pengeluaran individu dan kelas ExpenseFormatter untuk mengubah kueri pengguna menjadi data pengeluaran yang terstruktur.

 Setiap pengeluaran akan direpresentasikan dalam format:
 `{'date': '07-Mar-2025', 'description': 'flight to destination', 'amount': 675.99, 'category': 'Transportation'}`


In [ ]:
class Expense(BaseModel):
    date: str = Field(..., description="Date of expense in dd-MMM-yyyy format")
    description: str = Field(..., description="Expense description")
    amount: float = Field(..., description="Expense amount")
    category: str = Field(..., description="Expense category (e.g., Transportation, Meals, Accommodation, Miscellaneous)")

class ExpenseFormatter(BaseModel):
    raw_query: str = Field(..., description="Raw query input containing expense details")
    
    def parse_expenses(self) -> List[Expense]:
        """
        Parses the raw query into a list of Expense objects.
        Expected format: "date|description|amount|category" separated by semicolons.
        """
        expense_list = []
        for expense_str in self.raw_query.split(";"):
            if expense_str.strip():
                parts = expense_str.strip().split("|")
                if len(parts) == 4:
                    date, description, amount, category = parts
                    try:
                        expense = Expense(
                            date=date.strip(),
                            description=description.strip(),
                            amount=float(amount.strip()),
                            category=category.strip()
                        )
                        expense_list.append(expense)
                    except ValueError as e:
                        print(f"[LOG] Parse Error: Invalid data in '{expense_str}': {e}")
        return expense_list

## Mendefinisikan Alat - Menghasilkan Email

Buat fungsi alat untuk menghasilkan email guna mengajukan klaim pengeluaran.
- Alat ini menggunakan dekorator `@tool` dari Microsoft Agent Framework.
- Alat ini menghitung jumlah total pengeluaran dan memformat detailnya menjadi isi email.


In [ ]:
@tool(approval_mode="never_require")
def generate_expense_email(
    expense_data: Annotated[str, "Semicolon-separated expense entries in 'date|description|amount|category' format"]
) -> str:
    """Generate an email to submit an expense claim to the Finance Team."""
    formatter = ExpenseFormatter(raw_query=expense_data)
    expenses = formatter.parse_expenses()
    if not expenses:
        return "No valid expenses found to include in the email."
    total_amount = sum(e.amount for e in expenses)
    email_body = "Dear Finance Team,\n\n"
    email_body += "Please find below the details of my expense claim:\n\n"
    for e in expenses:
        email_body += f"- {e.date} | {e.description}: ${e.amount:.2f} ({e.category})\n"
    email_body += f"\nTotal Amount: ${total_amount:.2f}\n\n"
    email_body += "Receipts for all expenses are attached for your reference.\n\n"
    email_body += "Thank you,\n[Your Name]"
    return email_body

# Alat untuk Mengekstrak Biaya Perjalanan dari Gambar Struk

Buat fungsi alat untuk mengekstrak biaya perjalanan dari gambar struk.
- Alat ini menggunakan dekorator `@tool` dari Microsoft Agent Framework.
- Alat ini membaca gambar struk, mengenkodenya sebagai base64, dan mengembalikan data URI untuk dianalisis oleh agen.


In [ ]:
@tool(approval_mode="never_require")
def load_receipt_image(
    image_path: Annotated[str, "Path to the receipt image file"] = "receipt.jpg"
) -> str:
    """Load a receipt image and return its base64-encoded data URI for OCR extraction."""
    try:
        with open(image_path, "rb") as f:
            image_data = base64.b64encode(f.read()).decode("utf-8")
        return f"data:image/jpeg;base64,{image_data}"
    except Exception as e:
        error_msg = f"[LOG] Error loading image '{image_path}': {str(e)}"
        print(error_msg)
        return error_msg

## Memproses Pengeluaran

Tentukan agen dan hubungkan mereka ke dalam alur kerja berurutan menggunakan `WorkflowBuilder`.
- Agen OCR mengekstrak data pengeluaran terstruktur dari gambar struk menggunakan alat `load_receipt_image`.
- Agen Email mengambil data yang diekstrak dan menghasilkan email klaim pengeluaran yang profesional menggunakan alat `generate_expense_email`.
- `WorkflowBuilder` dengan `add_edge` membuat pipeline berurutan: Agen OCR → Agen Email.


In [ ]:
ocr_agent = client.as_agent(
    tools=[load_receipt_image],
    name="OCRAgent",
    instructions=(
        "You are an expert OCR assistant specialized in extracting structured data from receipt images. "
        "Use the 'load_receipt_image' tool to load the receipt image, then analyze it and extract "
        "travel-related expense details in the format: 'date|description|amount|category' separated by semicolons. "
        "Follow these rules: "
        "- Date: Convert dates (e.g., '4/4/22') to 'dd-MMM-yyyy' (e.g., '04-Apr-2022'). "
        "- Description: Extract item names. "
        "- Amount: Use numeric values (e.g., '4.50' from '$4.50'). "
        "- Category: Infer from context (e.g., 'Meals' for food, 'Transportation' for travel, "
        "'Accommodation' for lodging, 'Miscellaneous' otherwise). "
        "Ignore totals, subtotals, or service charges unless they are itemized expenses. "
        "If no expenses are found, return 'No expenses detected'. "
        "Return only the structured data, no additional text."
    ),
)

email_agent = client.as_agent(
    name="EmailAgent",
    tools=[generate_expense_email],
    instructions=(
        "You are an expense claim email generator. Take the travel expense data from the previous agent "
        "(in 'date|description|amount|category' format separated by semicolons) and use the "
        "'generate_expense_email' tool to produce a professional expense claim email. "
        "Pass the semicolon-separated expense data directly to the tool."
    ),
)

## Fungsi utama

Bangun alur kerja berurutan dan jalankan untuk memproses gambar struk dan menghasilkan email klaim pengeluaran.


> **Catatan:** Alur kerja ini saat ini mengirimkan gambar struk sebagai teks base64, yang sebagian besar model chat (termasuk gpt-4o) tidak akan memperlakukan sebagai gambar.  
> Ini juga mungkin melebihi jendela konteks model. Lebih baik menjalankan OCR dengan Azure AI Vision (atau alat OCR lain) dan hanya mengirimkan teks yang diekstrak, atau mengubah agar gambar dikirim sebagai pesan `image_url`.  
> Jika Anda hanya ingin menghindari kesalahan konteks, coba gunakan gambar struk yang lebih kecil atau model dengan jendela konteks yang lebih besar.


In [ ]:
workflow = WorkflowBuilder(start_executor=ocr_agent) \
    .add_edge(ocr_agent, email_agent) \
    .build()

prompt = (
    "Please extract the raw text from the receipt image at 'receipt.jpg', "
    "focusing on travel expenses like dates, descriptions, amounts, and categories "
    "(e.g., Transportation, Accommodation, Meals, Miscellaneous). "
    "Then generate a professional expense claim email."
)

last_author = None
events = workflow.run(
    prompt,
    stream=True,
)
async for event in events:
    if event.type == "output" and isinstance(event.data, AgentResponseUpdate):
        update = event.data
        author = update.author_name
        if author != last_author:
            if last_author is not None:
                print()
            print(f"\n{'='*50}")
            print(f"# Agent - {author}:")
            print(f"{'='*50}")
            last_author = author
        print(update.text, end="", flush=True)

---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Penafian**:
Dokumen ini telah diterjemahkan menggunakan layanan terjemahan AI [Co-op Translator](https://github.com/Azure/co-op-translator). Meskipun kami berupaya untuk mencapai akurasi, harap diketahui bahwa terjemahan otomatis mungkin mengandung kesalahan atau ketidakakuratan. Dokumen asli dalam bahasa aslinya harus dianggap sebagai sumber yang sah. Untuk informasi penting, disarankan menggunakan terjemahan profesional oleh manusia. Kami tidak bertanggung jawab atas kesalahpahaman atau penafsiran yang keliru yang timbul dari penggunaan terjemahan ini.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
